<a href="https://colab.research.google.com/github/Titantus/The-T0C-Predictive-Routing-Engine/blob/main/Lattice_Analysis_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# Calculate PSDs

# Enhanced version: Add modes + Pivot-like resonance
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def enhanced_saturation(steps=20000, pc=0.0497, noise=0.008, wg=1.0, ww=1.0):
    se = np.zeros(steps)
    Ec = np.zeros(steps)  # Coupling energy
    mode = np.zeros(steps)  # 0: Straight, 1: Loop
    resets = 0
    for i in range(1, steps):
        # Pivot Operator approximation
        phase_diff = np.cos(wg * i) * np.cos(ww * i)
        Ec[i] = 0.5 * phase_diff  # Simplified <Ec>

        delta = np.random.normal(0.0, noise) + 0.1 * Ec[i]
        se[i] = np.clip(se[i-1] + delta, 0.0, 1.0)

        if se[i] > 0.92:  # Coherence Horizon
            se[i] = pc * 2  # Reset toward equilibrium
            resets += 1
            mode[i] = 1
        else:
            mode[i] = 0 if np.abs(phase_diff) < 0.1 else 1  # Async -> Straight

    return se, Ec, mode, resets

se_enhanced, Ec_enhanced, mode_enhanced, resets_enhanced = enhanced_saturation()

freqs_se_data, psd_se_data = analyze_lattice_coherence(se_data)
freqs_se_enhanced, psd_se_enhanced = welch(se_enhanced, nperseg=2048)

# Set dark background style for plots
plt.style.use('dark_background')

# Create a multi-panel figure
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
fig.suptitle('Combined Simulation Analysis', fontsize=16)

# Plot 1: Se Saturation (Basic Model Short-term)
axes[0, 0].plot(se_data[:500])
axes[0, 0].axhline(0.95, color='r', linestyle='--', label='Clutch-Snap Threshold')
axes[0, 0].set_title('Se Saturation (Basic Model Short-term)')
axes[0, 0].legend()

# Plot 2: PSD (Basic Model Coherence Horizon)
axes[0, 1].loglog(freqs_se_data[1:], psd_se_data[1:])
axes[0, 1].set_title('PSD (Basic Model Coherence Horizon)')
axes[0, 1].set_xlabel('log(Frequency)')
axes[0, 1].set_ylabel('log(PSD)')

# Plot 3: Se Evolution (Enhanced Model Short-term)
axes[1, 0].plot(se_enhanced[:1000])
axes[1, 0].axhline(0.0497, color='r', ls='--')
axes[1, 0].set_title('Se Evolution (Enhanced Model Short-term)')

# Plot 4: PSD (Enhanced Model)
axes[1, 1].loglog(freqs_se_enhanced[1:], psd_se_enhanced[1:])
axes[1, 1].set_title('PSD (Enhanced Model)')
axes[1, 1].set_xlabel('log(Frequency)')
axes[1, 1].set_ylabel('log(PSD)')

# Plot 5: Coupling Energy (Ec Enhanced Model)
axes[2, 0].plot(Ec_enhanced[:1000])
axes[2, 0].set_title('Coupling Energy (Ec Enhanced Model)')

# Plot 6: Mode Distribution (Enhanced Model)
axes[2, 1].hist(mode_enhanced, bins=3)
axes[2, 1].set_title('Mode Distribution (Enhanced Model)')

plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent suptitle overlap
plt.show()

# Reset style to default for subsequent plots if needed
plt.style.use('default')
# Display numerical analysis results
alpha_phys, model_alpha, gap = quantify_alpha_gap()

print("--- Numerical Analysis Results ---")
print(f"Basic Model: Total Clutch-Snap events: {reset_count}")
print(f"Enhanced Model: Total Coherence Horizon resets: {resets_enhanced}")
print(f"Enhanced Model: Straight fraction: {np.mean(mode_enhanced==0):.3f}")
print(f"\nPhysical alpha (1/137.035999): {alpha_phys:.8f}")
print(f"Integer wrap alpha (1/137): {model_alpha:.8f}")
print(f"Geometric gap (lattice frustration): {gap:.8f}")

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt

def cycle_aware_pump(num_cycles=200, drive_freq_hz=45000, nx=140, ny=90, c0=1.0):
    dx = 1.0 / nx

    # Determine dt and steps based on num_cycles and drive_freq_hz
    period = 1.0 / drive_freq_hz
    total_time = num_cycles * period

    # 1. dt for temporal resolution (e.g., 20 points per cycle)
    N_points_per_period = 20
    dt_temp_res = period / N_points_per_period

    # 2. dt for spatial stability (Courant condition)
    dt_courant = 0.42 * dx / c0

    # Choose the smaller dt to satisfy both conditions
    dt = min(dt_temp_res, dt_courant)

    steps = int(total_time / dt)
    # Ensure at least one step if total_time is very small or dt is large
    if steps == 0:
        steps = 1
    # print(f"Running simulation for {steps} steps with dt={dt:.6f} for {num_cycles} cycles at {drive_freq_hz/1000:.1f} kHz") # Debug print

    # Tunable parameters (your sliders)
    diotic_weight = 0.48
    beta = 0.025          # nonlinearity for clustering
    drive_amp = 0.01      # Further reduced drive amplitude to encourage more complex dynamics
    strain_threshold = 0.038

    u = np.zeros((ny, nx))
    u_prev = np.zeros((ny, nx))

    x = np.linspace(0, 1, nx)
    y_grid = np.linspace(0, 1, ny)[:, None]
    width = 1.0 - (1.0 - 0.25) * x
    mask = y_grid < width[None, :]
    c_field = c0 * (1.0 + diotic_weight * (1.0 - y_grid / width[None, :]))

    r_history = []
    emf_history = []
    dump_events = 0
    dumping = False
    dump_timer = 0

    for t in range(steps):
        laplacian = (np.roll(u, -1, 0) + np.roll(u, 1, 0) +
                     np.roll(u, -1, 1) + np.roll(u, 1, 1) - 4*u) / dx**2

        u_new = 2*u - u_prev + (dt**2 * c_field**2 * laplacian * mask)

        # Dominant drive + noise (drive_freq_hz is now directly the physical frequency)
        noise = 0.12 * np.random.normal(0,1,ny) # Increased noise level
        drive = drive_amp * np.sin(2 * np.pi * drive_freq_hz * t * dt)
        u_new[:,0] += noise + drive

        u_new -= beta * (u_new ** 3)   # clustering nonlinearity

        # Phase order parameter (Kuramoto)
        u_dot = (u_new - u_prev) / (2*dt)
        grad_x = (np.roll(u_new,-1,1) - np.roll(u_new,1,1)) / (2*dx)
        phases = np.arctan2(u_dot[:,20:-20], c0*grad_x[:,20:-20] + 1e-8)  # mid-zone
        r_t = np.abs(np.mean(np.exp(1j * phases))) # Kuramoto order parameter
        r_history.append(r_t)

        # Shishiodoshi valve
        sink = x > 0.82
        apex_strain = np.mean(np.abs(u_new[:,sink]))
        emf = 0.0
        if apex_strain > strain_threshold and not dumping:
            dumping = True
            dump_timer = 12
            dump_events += 1
        if dumping:
            u_new[:,sink] *= 0.32
            emf = 6200 * (apex_strain / dt)   # scaled Faraday spike
            dump_timer -= 1
            if dump_timer <= 0:
                dumping = False
        emf_history.append(emf)

        u_prev, u = u, u_new

    return np.array(r_history), np.array(emf_history), dump_events

np.random.seed(42)
r, emf, dumps = cycle_aware_pump(num_cycles=200, drive_freq_hz=45000)

print(f"Mean R(t): {np.mean(r):.4f} | Max R: {np.max(r):.4f}")
print(f"Dump events: {dumps} | Peak EMF: {np.max(np.abs(emf)):.1f}")

In [ ]:
# @title
freq_range_khz_cycle_aware = np.linspace(44000, 45000, 20) # 44 kHz to 45 kHz
num_cycles_sweep = 2000 # Increased from 200 to allow more time for dynamics

mean_r_values_cycle_aware = []
dump_counts_cycle_aware = []

print(f"Running high-frequency sweep with cycle_aware_pump (num_cycles={num_cycles_sweep})...")
for freq_val_hz in freq_range_khz_cycle_aware:
    # Call the new cycle_aware_pump function
    r_temp, emf_temp, dumps_temp = cycle_aware_pump(num_cycles=num_cycles_sweep, drive_freq_hz=freq_val_hz)
    mean_r_values_cycle_aware.append(np.mean(r_temp))
    dump_counts_cycle_aware.append(dumps_temp)

# Plotting the resonance map for the cycle-aware high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_cycle_aware, (ax1_cycle_aware, ax2_cycle_aware) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Convert frequencies to kHz for plotting x-axis label
plot_freq_khz_cycle_aware = freq_range_khz_cycle_aware / 1000.0

ax1_cycle_aware.plot(plot_freq_khz_cycle_aware, mean_r_values_cycle_aware, marker='o', linestyle='-', color='red')
ax1_cycle_aware.set_ylabel('Mean R(t)')
ax1_cycle_aware.set_title(f'Resonance Map: Mean R(t) vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax1_cycle_aware.grid(True, linestyle='--', alpha=0.7)

ax2_cycle_aware.plot(plot_freq_khz_cycle_aware, dump_counts_cycle_aware, marker='o', linestyle='-', color='yellow')
ax2_cycle_aware.set_xlabel('Drive Frequency (kHz)')
ax2_cycle_aware.set_ylabel('Dump Events')
ax2_cycle_aware.set_title(f'Resonance Map: Dump Events vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax2_cycle_aware.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
freq_range_khz_cycle_aware = np.linspace(44000, 45000, 20) # 44 kHz to 45 kHz
num_cycles_sweep = 2000 # Increased from 200 to allow more time for dynamics

mean_r_values_cycle_aware = []
dump_counts_cycle_aware = []

print(f"Running high-frequency sweep with cycle_aware_pump (num_cycles={num_cycles_sweep})...")
for freq_val_hz in freq_range_khz_cycle_aware:
    # Call the new cycle_aware_pump function
    r_temp, emf_temp, dumps_temp = cycle_aware_pump(num_cycles=num_cycles_sweep, drive_freq_hz=freq_val_hz)
    mean_r_values_cycle_aware.append(np.mean(r_temp))
    dump_counts_cycle_aware.append(dumps_temp)

# Plotting the resonance map for the cycle-aware high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_cycle_aware, (ax1_cycle_aware, ax2_cycle_aware) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Convert frequencies to kHz for plotting x-axis label
plot_freq_khz_cycle_aware = freq_range_khz_cycle_aware / 1000.0

ax1_cycle_aware.plot(plot_freq_khz_cycle_aware, mean_r_values_cycle_aware, marker='o', linestyle='-', color='red')
ax1_cycle_aware.set_ylabel('Mean R(t)')
ax1_cycle_aware.set_title(f'Resonance Map: Mean R(t) vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax1_cycle_aware.grid(True, linestyle='--', alpha=0.7)

ax2_cycle_aware.plot(plot_freq_khz_cycle_aware, dump_counts_cycle_aware, marker='o', linestyle='-', color='yellow')
ax2_cycle_aware.set_xlabel('Drive Frequency (kHz)')
ax2_cycle_aware.set_ylabel('Dump Events')
ax2_cycle_aware.set_title(f'Resonance Map: Dump Events vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax2_cycle_aware.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
freq_range_khz_cycle_aware = np.linspace(44000, 45000, 20) # 44 kHz to 45 kHz
num_cycles_sweep = 2000 # Increased from 200 to allow more time for dynamics

mean_r_values_cycle_aware = []
dump_counts_cycle_aware = []

print(f"Running high-frequency sweep with cycle_aware_pump (num_cycles={num_cycles_sweep})...")
for freq_val_hz in freq_range_khz_cycle_aware:
    # Call the new cycle_aware_pump function
    r_temp, emf_temp, dumps_temp = cycle_aware_pump(num_cycles=num_cycles_sweep, drive_freq_hz=freq_val_hz)
    mean_r_values_cycle_aware.append(np.mean(r_temp))
    dump_counts_cycle_aware.append(dumps_temp)

# Plotting the resonance map for the cycle-aware high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_cycle_aware, (ax1_cycle_aware, ax2_cycle_aware) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Convert frequencies to kHz for plotting x-axis label
plot_freq_khz_cycle_aware = freq_range_khz_cycle_aware / 1000.0

ax1_cycle_aware.plot(plot_freq_khz_cycle_aware, mean_r_values_cycle_aware, marker='o', linestyle='-', color='red')
ax1_cycle_aware.set_ylabel('Mean R(t)')
ax1_cycle_aware.set_title(f'Resonance Map: Mean R(t) vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax1_cycle_aware.grid(True, linestyle='--', alpha=0.7)

ax2_cycle_aware.plot(plot_freq_khz_cycle_aware, dump_counts_cycle_aware, marker='o', linestyle='-', color='yellow')
ax2_cycle_aware.set_xlabel('Drive Frequency (kHz)')
ax2_cycle_aware.set_ylabel('Dump Events')
ax2_cycle_aware.set_title(f'Resonance Map: Dump Events vs. Drive Frequency ({num_cycles_sweep} Cycles, dt adjusted)')
ax2_cycle_aware.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
freq_range = np.linspace(0.02, 0.12, 20) # 20 points from 0.02 to 0.12
mean_r_values = []
dump_counts = []

for freq in freq_range:
    r_temp, emf_temp, dumps_temp = enhanced_kuramoto_pump(drive_freq=freq)
    mean_r_values.append(np.mean(r_temp))
    dump_counts.append(dumps_temp)

# Plotting the resonance map
plt.style.use('dark_background') # Maintain dark background style

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

ax1.plot(freq_range, mean_r_values, marker='o', linestyle='-', color='cyan')
ax1.set_ylabel('Mean R(t)')
ax1.set_title('Resonance Map: Mean R(t) vs. Drive Frequency')
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(freq_range, dump_counts, marker='o', linestyle='-', color='lime')
ax2.set_xlabel('Drive Frequency')
ax2.set_ylabel('Dump Events')
ax2.set_title('Resonance Map: Dump Events vs. Drive Frequency')
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
freq_range_khz = np.linspace(44.5, 45.5, 20) # 20 points from 44.5 to 45.5
mean_r_values_khz = []
dump_counts_khz = []

print("Running high-frequency sweep...")
for freq in freq_range_khz:
    # Using the same enhanced_kuramoto_pump function
    r_temp, emf_temp, dumps_temp = enhanced_kuramoto_pump(drive_freq=freq)
    mean_r_values_khz.append(np.mean(r_temp))
    dump_counts_khz.append(dumps_temp)

# Plotting the resonance map for high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_khz, (ax1_khz, ax2_khz) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

ax1_khz.plot(freq_range_khz, mean_r_values_khz, marker='o', linestyle='-', color='orange')
ax1_khz.set_ylabel('Mean R(t)')
ax1_khz.set_title('Resonance Map: Mean R(t) vs. Drive Frequency (44.5-45.5 kHz Range)')
ax1_khz.grid(True, linestyle='--', alpha=0.7)

ax2_khz.plot(freq_range_khz, dump_counts_khz, marker='o', linestyle='-', color='magenta')
ax2_khz.set_xlabel('Drive Frequency (kHz)')
ax2_khz.set_ylabel('Dump Events')
ax2_khz.set_title('Resonance Map: Dump Events vs. Drive Frequency (44.5-45.5 kHz Range)')
ax2_khz.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
scaled_drive_freq_range = np.linspace(0.0445, 0.0455, 20) # Corresponds to 44.5-45.5 kHz with freq_scale=1e3
freq_scale_val = 1e3 # Scaling factor for kHz

mean_r_values_scaled = []
dump_counts_scaled = []

print(f"Running scaled high-frequency sweep with freq_scale={freq_scale_val}...")
for drive_f in scaled_drive_freq_range:
    # Call the modified function with freq_scale
    r_temp, emf_temp, dumps_temp = enhanced_kuramoto_pump(drive_freq=drive_f, freq_scale=freq_scale_val)
    mean_r_values_scaled.append(np.mean(r_temp))
    dump_counts_scaled.append(dumps_temp)

# Plotting the resonance map for scaled high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_scaled, (ax1_scaled, ax2_scaled) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Convert drive_f to actual kHz values for plotting x-axis
actual_freq_khz = scaled_drive_freq_range * freq_scale_val

ax1_scaled.plot(actual_freq_khz, mean_r_values_scaled, marker='o', linestyle='-', color='red')
ax1_scaled.set_ylabel('Mean R(t)')
ax1_scaled.set_title(f'Resonance Map: Mean R(t) vs. Drive Frequency (Scaled to {freq_scale_val/1e3}kHz Range)')
ax1_scaled.grid(True, linestyle='--', alpha=0.7)

ax2_scaled.plot(actual_freq_khz, dump_counts_scaled, marker='o', linestyle='-', color='yellow')
ax2_scaled.set_xlabel('Drive Frequency (Hz, scaled)')
ax2_scaled.set_ylabel('Dump Events')
ax2_scaled.set_title(f'Resonance Map: Dump Events vs. Drive Frequency (Scaled to {freq_scale_val/1e3}kHz Range)')
ax2_scaled.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default

In [ ]:
# @title
freq_range_hz_actual = np.linspace(44500, 45500, 20) # 44.5 kHz to 45.5 kHz in Hz

mean_r_values_khz_new = []
dump_counts_khz_new = []

print("Running new high-frequency sweep with dynamically adjusted dt...")
for freq_val_hz in freq_range_hz_actual:
    # Call the modified function with drive_freq_hz directly
    r_temp, emf_temp, dumps_temp = enhanced_kuramoto_pump(drive_freq_hz=freq_val_hz)
    mean_r_values_khz_new.append(np.mean(r_temp))
    dump_counts_khz_new.append(dumps_temp)

# Plotting the resonance map for new high frequencies
plt.style.use('dark_background') # Maintain dark background style

fig_new_khz, (ax1_new_khz, ax2_new_khz) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Convert frequencies to kHz for plotting x-axis label
plot_freq_khz = freq_range_hz_actual / 1000.0

ax1_new_khz.plot(plot_freq_khz, mean_r_values_khz_new, marker='o', linestyle='-', color='red')
ax1_new_khz.set_ylabel('Mean R(t)')
ax1_new_khz.set_title('Resonance Map: Mean R(t) vs. Drive Frequency (44.5-45.5 kHz Range, dt adjusted)')
ax1_new_khz.grid(True, linestyle='--', alpha=0.7)

ax2_new_khz.plot(plot_freq_khz, dump_counts_khz_new, marker='o', linestyle='-', color='yellow')
ax2_new_khz.set_xlabel('Drive Frequency (kHz)')
ax2_new_khz.set_ylabel('Dump Events')
ax2_new_khz.set_title('Resonance Map: Dump Events vs. Drive Frequency (44.5-45.5 kHz Range, dt adjusted)')
ax2_new_khz.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

plt.style.use('default') # Reset style to default